# SevenNet-Polar · Born effective charges: CPU / free T4 quickstart

Train a small equivariant model on **real ZrO₂ DFT/DFPT data**, reload its checkpoint, and examine convergence and held-out parity. The loss targets BECs only; energy and force outputs are untrained and should not be used.

**Colab:** choose Runtime → Change runtime type → T4 (optional; CPU also works), then Runtime → Run all. No paid plan or FlashTP build is required. Local Jupyter needs PyTorch installed first. The notebook preserves that installation.

The compact model is an educational example, not a production potential. No dielectric model is trained. The [tutorial guide and validation records](https://github.com/AugustinLu/SevenNet-Polar/tree/tutorials/colab-bec/tutorials/colab) explain current test coverage.

## 1. Install the Polar fork and inspect the runtime

The package import is called `sevenn`, as in upstream SevenNet. Setup installs this repository and verifies its origin. A fresh Colab session already includes PyTorch. Installation output is saved to a file; a failure prints its final lines. All runs use float32, including T4 (no BF16 assumptions).

In [ ]:
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time
import uuid

START = time.perf_counter()
REPOSITORY = 'https://github.com/AugustinLu/SevenNet-Polar.git'
REPO_REF = (
    'tutorials/colab-bec'  # Preview branch; report.json records the exact commit.
)

# Local Jupyter reuses a checkout. Colab fetches the tutorial branch.
local = os.environ.get('SEVENNET_TUTORIAL_REPO')
candidates = (
    ([Path(local)] if local else [])
    + list(Path.cwd().resolve().parents)
    + [Path.cwd(), Path.cwd() / 'SevenNet-Polar']
)
REPO = next(
    (
        p.resolve()
        for p in candidates
        if (p / 'tutorials/colab/support.py').is_file()
    ),
    None,
)
fetch_started = time.perf_counter()
if REPO is None:
    REPO = Path.cwd() / ('SevenNet-Polar-' + uuid.uuid4().hex[:8])
    subprocess.run(
        [
            'git',
            'clone',
            '--depth',
            '1',
            '--branch',
            REPO_REF,
            REPOSITORY,
            str(REPO),
        ],
        check=True,
    )
FETCH_SECONDS = time.perf_counter() - fetch_started


TORCH_BEFORE = importlib.metadata.version('torch')
WORK = Path.cwd() / ('polar-tutorial-' + uuid.uuid4().hex[:8])
WORK.mkdir()
INSTALL_LOG = WORK / 'installation.log'
# Constrain PyTorch to the runtime wheel. Pip retains already-satisfied dependencies.
constraint = WORK / 'runtime-constraint.txt'
constraint.write_text('torch==' + TORCH_BEFORE + '\n')
install_started = time.perf_counter()
SKIP_INSTALL = (
    os.environ.get('SEVENNET_TUTORIAL_SKIP_INSTALL') == '1'
)  # Test harness only.
if not SKIP_INSTALL:
    with INSTALL_LOG.open('w') as log:
        result = subprocess.run(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--constraint',
                str(constraint),
                '--editable',
                str(REPO),
                'matplotlib',
            ],
            stdout=log,
            stderr=subprocess.STDOUT,
        )
    if result.returncode:
        print(INSTALL_LOG.read_text()[-5000:])
        raise RuntimeError(f'Installation failed. Full log: {INSTALL_LOG}')
INSTALL_SECONDS = time.perf_counter() - install_started
if importlib.metadata.version('torch') != TORCH_BEFORE:
    raise RuntimeError('PyTorch changed. Restart the runtime before continuing.')
if (
    'sevenn' in sys.modules
    and REPO not in Path(sys.modules['sevenn'].__file__).resolve().parents
):
    raise RuntimeError(
        'Another sevenn was already imported. '
        'Restart the kernel and run this notebook first.'
    )
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'tutorials/colab'))
import support
from IPython.display import display

ENVIRONMENT = support.environment(REPO)
DEVICE = ENVIRONMENT['device']
print(json.dumps(ENVIRONMENT, indent=2))
print(
    f'Fetch: {FETCH_SECONDS:.1f} s · install: {INSTALL_SECONDS:.1f} s '
    '· PyTorch retained'
)
print(f'Artifacts: {WORK}')
if DEVICE == 'cpu':
    print(
        'CPU selected. For Colab GPU: Runtime → Change runtime type → '
        'T4 or A100, then Run all.'
    )

In [ ]:
FLASH_REPORT = {'status': 'not requested; standard e3nn quickstart'}

## 2. Real labels and an explicit split

The 30 public example structures come from the author's [SevenNet-Polar datasets](https://doi.org/10.5281/zenodo.21322761), described in [Lu et al. (2026), §4.7](https://arxiv.org/html/2607.14827v3#S4.SS7). They contain pristine and oxygen-deficient cells. We reuse the small file already shipped with the repository; no large archive download is required.

We split by **whole structure**, stratifying compositions into 20 training, 5 validation, and 5 test cells. Test data never enters training, normalization statistics, or checkpoint selection. Exact geometry duplicate screening and E/F/S/BEC serialization checks run below. Frame/trajectory IDs are absent, so correlated configurations may span partitions: these small-sample errors demonstrate the workflow, not independent generalization. For research, split by trajectory/parent structure and reserve a separate test dataset.

BEC labels have shape `(atoms, 3, 3)` in units of elementary charge **e**. The first tensor index is polarization/field direction and the second is displacement direction. The raw label file is retained in the repository; tutorial copies explicitly select only E/F/S/BEC.

In [ ]:
DATA_DIR = WORK / 'data'
structures, split_paths, all_path, split_manifest, split_summary = (
    support.prepare_data(REPO, DATA_DIR, seed=42)
)
display(split_summary)
print(
    f'{len(structures)} structures; '
    f'{min(map(len, structures))}–{max(map(len, structures))} atoms per cell'
)
print('Source SHA256:', split_manifest['source_sha256'])
print('BEC and E/F/S label round trips passed. Duplicate geometry screen passed.')

## 3. Configure the model

`lmax=2` retains tensor information; 16 channels and 3 message-passing layers keep this exercise small. The existing CLI still computes energy derivatives even with zero energy/force loss weights. CPU preprocessing uses one worker for portability. Each rerun creates a new output directory.

The printed YAML is the actual training input. Only training data supplies shift and neighbor statistics. Checkpoint selection minimizes the weighted validation `TotalLoss`.

In [ ]:
# Adjust these few controls; keep the test split fixed.
EPOCHS = int(os.environ.get('SEVENNET_TUTORIAL_EPOCHS', '20'))
BATCH_SIZE = 1
CHANNELS = 16
LAYERS = 3
SEED = 42
MULTITASK = False
CONFIG = support.make_config(
    split_paths,
    EPOCHS,
    multitask=MULTITASK,
    batch_size=BATCH_SIZE,
    channel=CHANNELS,
    layers=LAYERS,
    device=DEVICE,
    seed=SEED,
)
import yaml

print(yaml.safe_dump(CONFIG, sort_keys=False))

## 4. Train and inspect convergence

Training uses `sevenn train` through a profiling wrapper. Progress appears about every 30 seconds; full logs and epoch CSVs are retained. Wall time includes preprocessing, model construction and checkpoint writing. GPU memory is the process's peak PyTorch allocation/reservation, not total driver memory. A missing GPU uses CPU.

Force/stress convergence plots use component RMSE, matching the parity tables. The raw CLI CSV uses vector/tensor norm RMSE: divide force by √3 and stress by 10√6 to obtain component RMSE in eV/Å and GPa. Training curves accumulate predictions while weights change; parity uses the fixed best checkpoint.

In [ ]:
RUN = WORK / 'training'
CHECKPOINT, TRAIN_PROFILE = support.train(REPO, RUN, CONFIG)
print('Best validation checkpoint:', CHECKPOINT.name)
print(json.dumps(TRAIN_PROFILE, indent=2))

In [ ]:
curve, convergence_figure = support.learning_curves(RUN, multitask=MULTITASK)
display(curve.tail())
display(convergence_figure)
import matplotlib.pyplot as plt

plt.close(convergence_figure)

## 5. Reload and evaluate the chosen checkpoint

The inference CLI reloads the **best validation checkpoint** and predicts the original frame order. Training, validation, and held-out test points have different colors. Tables report component-weighted RMSE and MAE; tensor components from one atom are not independent samples.

Diagonal and all six off-diagonal BEC entries are reported separately; BEC tensors need not be symmetric.

Energy and force parity is intentionally omitted for this BEC-only fit.

In [ ]:
atom_predictions, graph_predictions, INFERENCE_PROFILE = support.evaluate(
    REPO, RUN, CHECKPOINT, all_path, DEVICE, batch_size=BATCH_SIZE
)
metrics, parity_figure = support.metrics_and_parity(
    atom_predictions, graph_predictions, split_manifest, RUN, multitask=MULTITASK
)
display(parity_figure)
plt.close(parity_figure)
display(metrics.round(5))
FIRST_PARITY_SECONDS = time.perf_counter() - START
print(
    f'Setup to first parity: {FIRST_PARITY_SECONDS:.1f} s '
    '(runtime allocation excluded)'
)

## 6. Use the checkpoint from Python and inspect the sum rule

`SevenNetCalculator` reloads the checkpoint for a test structure and is checked against the CLI. The table reports the maximum absolute component of **Σᵢ Z*ᵢ**, in e. These are raw predictions and raw DFT labels: no acoustic-sum-rule correction is applied. A small BEC RMSE alone does not guarantee a small cell-wide residual.

In [ ]:
structure_id = split_manifest['indices']['Test'][0]
predicted_bec = support.reload_ase(
    CHECKPOINT, structures[structure_id], DEVICE, atom_predictions, structure_id
)
print('ASE checkpoint reload agrees with CLI predictions.')
print(f"First atom's Z* in test structure {structure_id} (e):\n{predicted_bec[0]}")
asr = support.asr_diagnostics(atom_predictions, split_manifest, RUN)
display(
    asr.groupby('partition')[
        ['reference_max_abs_sum_e', 'prediction_max_abs_sum_e']
    ].max()
)

## Export and continue

Download the archive to keep the model, exact configuration, split data, metrics, PNG/PDF parity figures, logs, and machine-readable timings. Colab runtime storage is temporary. To use your own data, provide whole-structure train/validation/test extxyz files and keep their DFT conventions consistent. Expand the training set and revalidate before using a model for research.

In [ ]:
REPORT = {
    'environment': ENVIRONMENT,
    'install_skipped': SKIP_INSTALL,
    'timings': {
        'repository_fetch_seconds': FETCH_SECONDS,
        'package_install_seconds': INSTALL_SECONDS,
        'setup_to_first_parity_seconds': FIRST_PARITY_SECONDS,
    },
    'training': TRAIN_PROFILE,
    'inference': INFERENCE_PROFILE,
    'split': split_manifest,
    'configuration': CONFIG,
    'flash': FLASH_REPORT,
}
ARCHIVE = support.export_artifacts(RUN, DATA_DIR, REPORT)
print(
    'Saved checkpoint, YAML, data/split, logs, metrics, figures, environment:',
    ARCHIVE,
)
DOWNLOAD = False  # Set True and rerun to download from Colab.
if DOWNLOAD:
    try:
        from google.colab import files

        files.download(str(ARCHIVE))
    except ImportError:
        print('In local Jupyter, open the archive at the path printed above.')

## References and troubleshooting

Please cite [SevenNet-Polar, Lu et al. (2026)](https://arxiv.org/abs/2607.14827), the [dataset archive](https://doi.org/10.5281/zenodo.21322761), and [SevenNet, Park et al. (2024)](https://doi.org/10.1021/acs.jctc.4c00190). Accelerator users should also cite [FlashTP, Lee et al. (2025)](https://openreview.net/forum?id=wiQe95BPaB). The repository preserves the Polar and upstream MIT licenses; the data archive provides its own license.

- **Wrong package origin:** restart the kernel and run setup before importing `sevenn`.
- **No GPU / different GPU:** use the detected standard backend; record the actual device. Colab allocation varies.
- **Out of memory:** reduce `BATCH_SIZE` to 1 and rerun in a fresh session.
- **FlashTP build or numerical failure:** keep `TRY_FLASHTP=False`; inspect `flash_build.log` and `flash_report.json`. No acceleration result is claimed after a failed check.
- **Slow CPU:** reduce epochs for a smoke test; interpret its accuracy accordingly. Check [measured validation records](https://github.com/AugustinLu/SevenNet-Polar/blob/tutorials/colab-bec/tutorials/colab/VALIDATION.md) before estimating GPU time.